In [1]:
# %%
# =============================================================================
# Paper C · 07_case_study  (qualitative illustration of RQ2 contagion)
# Two visualizations that show HOW relational risk propagates:
#   (A) CONTRAST cases — firms the intra-only model (S0) got WRONG but the
#       contagion model (S1) got RIGHT. Isolates the pure effect of propagation.
#   (B) NETWORK view  — a bankrupt firm's relational neighborhood (co-lawsuit,
#       investment, supervision links), colored by label, to show the structure
#       through which risk spreads.
#
# Uses idx2name (smesd_idmaps.pt) for real names; ANONYMIZE=True masks them to
# C###/P### for the paper figures (Chinese real names off by default).
#
# Inputs : smesd_hetero.pt, smesd_idmaps.pt, SMEsD pickles (for edges+events)
# Outputs: artifacts/figures/07_contrast_cases.png, 07_network_<id>.png
# =============================================================================
import pickle, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
DATA_DIR = ROOT / "data" / "external" / "ComRisk" / "data"
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "artifacts" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
SEED = 42
ANONYMIZE = True          # True → C###/P### labels; False → real names
torch.manual_seed(SEED); np.random.seed(SEED)


# %%
# --- 1. Load graph, id maps, raw edges ---------------------------------------
data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
N_COMP = comp.num_nodes

idmaps = torch.load(PROCESSED / "smesd_idmaps.pt", weights_only=False)
comp_global = idmaps["comp_global"]
idx2name = idmaps["idx2name"]
g2l = {int(g): i for i, g in enumerate(comp_global)}        # global -> local company
l2g = {i: int(g) for i, g in enumerate(comp_global)}        # local company -> global

# relation names (SMEsD.md) for readable edge labels
REL_NAME = {0: "supervise", 1: "supervise", 2: "execute", 3: "execute",
            4: "stakeholder", 5: "stakeholder", 6: "invest_CC", 7: "invest_CC",
            8: "invest_CP", 9: "invest_CP", 10: "branch", 11: "branch",
            12: "loan", 13: "deal", 14: "rent", 15: "creditor",
            16: "loan", 17: "deal", 18: "rent", 19: "creditor"}

def load(n):
    with open(DATA_DIR / n, "rb") as f:
        return pickle.load(f)

# global-id edge list with relation number (for neighborhood extraction)
edges_global = []       # (src_global, dst_global, rel_num)
for fn in ["train_data.pkl", "validate_data.pkl", "test_data.pkl"]:
    hg = load(fn)[2]
    ei = np.asarray(hg[0]); et = np.asarray(hg[1])
    if ei.shape[0] != 2: ei = ei.T
    for k in range(ei.shape[1]):
        edges_global.append((int(ei[0, k]), int(ei[1, k]), int(et[k])))
edges_global = list(set(edges_global))
print(f"loaded {len(edges_global):,} global edges")


# %%
# --- 2. Rebuild S0 (intra MLP) and S1 (contagion GNN) to get predictions -----
# Minimal re-train (seed 42) mirroring 06, then capture per-node test probs.
from sklearn.preprocessing import StandardScaler
from torch_geometric.nn import HeteroConv, SAGEConv

Xc = comp.x.numpy().astype(np.float64)
Xc[:, :2] = np.log1p(np.clip(Xc[:, :2], 0, None))
tr, va, te = comp.train_mask, comp.val_mask, comp.test_mask
y = comp.y
Xs = torch.tensor(StandardScaler().fit(Xc[tr.numpy()]).transform(Xc).astype(np.float32))

EDGES_0_11 = [et for et in data.edge_types if int(et[1].split("_")[1]) <= 11]

class MLP(nn.Module):
    def __init__(self, d, h=64, o=2, p=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(h, h), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(h, o))
    def forward(self, x): return self.net(x)

class HeteroGNN(nn.Module):
    def __init__(self, in_dims, ets, h=64, o=2, L=2, p=0.3):
        super().__init__()
        self.drop = p
        self.lin_in = nn.ModuleDict({nt: nn.Linear(d, h) for nt, d in in_dims.items()})
        self.convs = nn.ModuleList([HeteroConv({et: SAGEConv((h, h), h) for et in ets},
                                               aggr="sum") for _ in range(L)])
        self.head = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Dropout(p), nn.Linear(h, o))
    def forward(self, xd, eid):
        hh = {nt: F.relu(self.lin_in[nt](x)) for nt, x in xd.items()}
        for c in self.convs:
            hh = c(hh, eid)
            hh = {nt: F.dropout(F.relu(v), p=self.drop, training=self.training)
                  for nt, v in hh.items()}
        return self.head(hh["company"])

def cls_w(y, tr):
    c = torch.bincount(y[tr], minlength=2).float()
    return c.sum() / (2 * c.clamp(min=1))

def train_model(model, forward_fn, epochs, patience=35, lr=5e-3, wd=5e-4):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    w = cls_w(y, tr); best, state, wait = -1, None, 0
    from sklearn.metrics import roc_auc_score
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(forward_fn(model)[tr], y[tr], weight=w)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = F.softmax(forward_fn(model), 1)[:, 1].numpy()
        v = roc_auc_score(y[va].numpy(), p[va.numpy()])
        if v > best: best, state, wait = v, {k: vv.clone() for k, vv in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(state); model.eval()
    with torch.no_grad():
        return F.softmax(forward_fn(model), 1)[:, 1].numpy()

x_dict = {"company": Xs, "person": data["person"].x}
ei_dict = {et: data[et].edge_index for et in EDGES_0_11}
in_dims = {"company": Xs.size(1), "person": data["person"].x.size(1)}

p_S0 = train_model(MLP(Xs.size(1)), lambda m: m(Xs), epochs=300)
p_S1 = train_model(HeteroGNN(in_dims, EDGES_0_11),
                   lambda m: m(x_dict, ei_dict), epochs=200)
print("S0 (intra) and S1 (contagion) predictions ready")


# %%
# --- 3. Find CONTRAST cases: S0 wrong, S1 right, on bankrupt test firms -------
te_idx = np.where(te.numpy())[0]
yv = y.numpy()
s0_pred = (p_S0 >= 0.5).astype(int)
s1_pred = (p_S1 >= 0.5).astype(int)

# bankrupt firms (y=1) that S0 missed but S1 caught
contrast = [i for i in te_idx
            if yv[i] == 1 and s0_pred[i] == 0 and s1_pred[i] == 1]
# rank by how decisively contagion flipped it (S1 conf − S0 conf)
contrast.sort(key=lambda i: (p_S1[i] - p_S0[i]), reverse=True)
print(f"contrast cases (bankrupt, S0 wrong → S1 right): {len(contrast)}")

def label_for(global_id, is_company=True):
    if ANONYMIZE:
        return f"C{global_id}" if is_company else f"P{global_id}"
    return idx2name.get(global_id, f"{'C' if is_company else 'P'}{global_id}")

rows = []
for i in contrast[:10]:
    g = l2g[i]
    rows.append({"local_idx": i, "global_id": g, "name": label_for(g),
                 "S0_prob": round(float(p_S0[i]), 3),
                 "S1_prob": round(float(p_S1[i]), 3),
                 "flip": round(float(p_S1[i] - p_S0[i]), 3)})
contrast_df = pd.DataFrame(rows)
print(contrast_df.to_string(index=False))


# %%
# --- 4. FIGURE A: contrast bar chart (S0 vs S1 probability per case) ----------
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False

topk = contrast_df.head(8)
fig, ax = plt.subplots(figsize=(9, 5))
ypos = np.arange(len(topk))
ax.barh(ypos - 0.2, topk["S0_prob"], height=0.4, label="S0 intra-only",
        color="#bbbbbb")
ax.barh(ypos + 0.2, topk["S1_prob"], height=0.4, label="S1 +contagion",
        color="#c0392b")
ax.axvline(0.5, ls="--", c="k", lw=1, label="decision threshold")
ax.set_yticks(ypos); ax.set_yticklabels(topk["name"])
ax.set_xlabel("predicted bankruptcy probability")
ax.set_title("Contagion flips missed bankruptcies\n"
             "(true-bankrupt firms: intra-only missed, +contagion caught)")
ax.legend(loc="lower right"); ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(FIGURES / "07_contrast_cases.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"→ saved {FIGURES / '07_contrast_cases.png'}")


# %%
# --- 5. Build neighborhood of the top contrast firm --------------------------
# 1-hop (optionally 2-hop) relational neighborhood, with labels/relations.
def neighborhood(center_global, hops=1):
    nodes = {center_global}
    frontier = {center_global}
    edge_list = []
    for _ in range(hops):
        nxt = set()
        for (s, d, r) in edges_global:
            if s in frontier or d in frontier:
                edge_list.append((s, d, r))
                nxt.update([s, d])
        nodes |= nxt
        frontier = nxt - nodes if False else nxt
    # dedup edges
    edge_list = list(set(edge_list))
    return nodes, edge_list

center = int(contrast_df.iloc[0]["global_id"])
nodes, elist = neighborhood(center, hops=1)
print(f"center {label_for(center)}: {len(nodes)} nodes, {len(elist)} edges (1-hop)")


# %%
# --- 6. FIGURE B: network visualization --------------------------------------
# Company nodes colored by label (red=bankrupt, green=survived, grey=unlabeled/
# not in company set); person nodes as light squares. Center ringed.
import networkx as nx

G = nx.MultiDiGraph()
for n in nodes:
    is_comp = n in g2l
    lab = None
    if is_comp:
        li = g2l[n]
        lab = int(yv[li]) if (tr[li] or va[li] or te[li]) else -1
    G.add_node(n, is_company=is_comp, label=lab)
for (s, d, r) in elist:
    G.add_edge(s, d, rel=REL_NAME.get(r, str(r)))

def node_color(n):
    if not G.nodes[n]["is_company"]:
        return "#d9e6f2"                       # person
    lab = G.nodes[n]["label"]
    return {1: "#e74c3c", 0: "#2ecc71"}.get(lab, "#cccccc")  # bankrupt/survive/unk

pos = nx.spring_layout(G, seed=SEED, k=0.6)
fig, ax = plt.subplots(figsize=(10, 8))
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.3, arrows=False, width=1)
nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=[node_color(n) for n in G.nodes],
    node_shape="o",
    node_size=[520 if n == center else (240 if n in g2l else 120) for n in G.nodes],
    edgecolors=["black" if n == center else "none" for n in G.nodes],
    linewidths=[2 if n == center else 0 for n in G.nodes])
nx.draw_networkx_labels(G, pos, ax=ax,
    labels={n: label_for(n, n in g2l) for n in G.nodes}, font_size=7)
# edge relation labels (dedup per pair for readability)
elabels = {}
for u, v, dd in G.edges(data=True):
    elabels.setdefault((u, v), dd["rel"])
nx.draw_networkx_edge_labels(G, pos, ax=ax, edge_labels=elabels, font_size=6,
                             label_pos=0.5, alpha=0.7)
ax.set_title(f"Relational neighborhood of {label_for(center)} (bankrupt)\n"
             "red=bankrupt · green=survived · grey=unlabeled · blue=person · "
             "ringed=center", fontsize=11)
ax.axis("off"); plt.tight_layout()
out = FIGURES / f"07_network_{center}.png"
plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
print(f"→ saved {out}")


# %%
# --- 7. Neighborhood risk summary (does the center sit among bankrupt peers?) -
comp_neighbors = [n for n in nodes if n in g2l and n != center]
labeled = [(n, G.nodes[n]["label"]) for n in comp_neighbors if G.nodes[n]["label"] in (0, 1)]
if labeled:
    bank = sum(1 for _, l in labeled if l == 1)
    print(f"\ncenter {label_for(center)} neighborhood:")
    print(f"  labeled company neighbors: {len(labeled)}  "
          f"bankrupt: {bank} ({bank/len(labeled):.0%})")
    print(f"  → illustrates contagion: risk concentrated among connected firms")


# %%
# --- 8. Notes ----------------------------------------------------------------
print("""
07 complete — qualitative contagion illustration
================================================
  Figure A (07_contrast_cases.png): true-bankrupt firms the intra-only model
    missed but the contagion model caught — the per-case effect of propagation.
  Figure B (07_network_<id>.png): the top case's relational neighborhood; if the
    center sits among bankrupt peers via invest/supervise/co-lawsuit links, it
    visually explains WHY contagion (RQ2) works.

  Set ANONYMIZE=False only for private inspection; keep True for the paper to
  avoid publishing real Chinese company names.

  Paper C narrative now complete:
    RQ1 graph beats basic floor · RQ2 contagion is the core signal (+0.089***) ·
    RQ3 per-lawsuit LLM enrichment adds no robust gain — value is relational.
""")
print("07_case_study complete.")

loaded 25,952 global edges
S0 (intra) and S1 (contagion) predictions ready
contrast cases (bankrupt, S0 wrong → S1 right): 43
 local_idx  global_id  name  S0_prob  S1_prob  flip
      1562       1562 C1562    0.286    0.999 0.713
       270        270  C270    0.319    0.971 0.652
       143        143  C143    0.357    0.991 0.634
       115        115  C115    0.175    0.723 0.548
       176        176  C176    0.421    0.907 0.486
      1044       1044 C1044    0.458    0.892 0.434
       271        271  C271    0.246    0.644 0.398
       322        322  C322    0.242    0.635 0.393
       489        489  C489    0.404    0.793 0.389
      2751       2751 C2751    0.256    0.638 0.382
→ saved C:\Users\miy\Downloads\1000_논문투고\1_양문일_교신저자\32_연구기획4개\30_연구기획4개\artifacts\figures\07_contrast_cases.png
center C1562: 4 nodes, 6 edges (1-hop)
→ saved C:\Users\miy\Downloads\1000_논문투고\1_양문일_교신저자\32_연구기획4개\30_연구기획4개\artifacts\figures\07_network_1562.png

center C1562 neighborhood:
  labeled com

In [2]:
# %%
# --- 5. Pick a NEIGHBORHOOD-RICH contrast case + adaptive hop -----------------
# Figure A ranks by flip (dramatic reversal). Figure B needs relational STRUCTURE,
# so here we re-rank the 43 contrast cases by how many LABELED company neighbors
# they have, then draw 1-hop; if still sparse, expand to 2-hop capped at 40 nodes.
MAX_NODES = 40

def neighborhood(center_global, hops=1, max_nodes=MAX_NODES):
    """BFS out to `hops`, but stop adding once max_nodes is reached (invest
    relations are dense — 2-hop can explode without a cap)."""
    nodes = {center_global}
    frontier = {center_global}
    edge_list = []
    for _ in range(hops):
        nxt = set()
        for (s, d, r) in edges_global:
            if s in frontier or d in frontier:
                edge_list.append((s, d, r))
                nxt.update([s, d])
        # cap growth: keep company neighbors first (more informative than persons)
        new_nodes = nxt - nodes
        if len(nodes) + len(new_nodes) > max_nodes:
            comp_first = sorted(new_nodes, key=lambda n: (n not in g2l, n))
            new_nodes = set(comp_first[:max_nodes - len(nodes)])
        nodes |= new_nodes
        frontier = new_nodes
        if len(nodes) >= max_nodes:
            break
    # keep only edges whose endpoints are both retained
    edge_list = list({(s, d, r) for (s, d, r) in edge_list
                      if s in nodes and d in nodes})
    return nodes, edge_list

def count_labeled_company_neighbors(center_global, hops=1):
    n, _ = neighborhood(center_global, hops=hops)
    labeled = 0
    for x in n:
        if x != center_global and x in g2l:
            li = g2l[x]
            if tr[li] or va[li] or te[li]:
                labeled += 1
    return labeled

# rank contrast cases by 1-hop labeled-company-neighbor count (desc)
ranked = sorted(
    contrast_df["global_id"].astype(int).tolist(),
    key=lambda g: count_labeled_company_neighbors(g, hops=1),
    reverse=True)

center = ranked[0]
n1 = count_labeled_company_neighbors(center, hops=1)
print(f"neighborhood-richest contrast case: {label_for(center)} "
      f"({n1} labeled company neighbors at 1-hop)")

# adaptive: 1-hop if it already shows structure (>=3 labeled co.), else 2-hop
if n1 >= 3:
    hops = 1
else:
    hops = 2
    print(f"  1-hop sparse (<3) → expanding to 2-hop (cap {MAX_NODES} nodes)")

nodes, elist = neighborhood(center, hops=hops)
n_comp_nodes = sum(1 for n in nodes if n in g2l)
print(f"center {label_for(center)}: {len(nodes)} nodes "
      f"({n_comp_nodes} companies, {len(nodes)-n_comp_nodes} persons), "
      f"{len(elist)} edges ({hops}-hop)")

neighborhood-richest contrast case: C1562 (1 labeled company neighbors at 1-hop)
  1-hop sparse (<3) → expanding to 2-hop (cap 40 nodes)
center C1562: 14 nodes (6 companies, 8 persons), 52 edges (2-hop)


In [29]:
# %%
# --- Figure export setup: create paper_C folder + save helper (run once) -----
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
PAPER_DIR = ROOT / "paper_C"
PAPER_DIR.mkdir(parents=True, exist_ok=True)

# Embed fonts as text (avoids font substitution at typesetting). EPS has no alpha.
matplotlib.rcParams.update({
    "ps.fonttype": 42, "pdf.fonttype": 42, "axes.unicode_minus": False,
})

def save_fig(fig, name, dpi=600):
    """Save as PDF + EPS + PNG (caption-free). Does not print the path."""
    for ext in ("pdf", "eps", "png"):
        fig.savefig(PAPER_DIR / f"{name}.{ext}", format=ext, dpi=dpi,
                    bbox_inches="tight")
    print(f"saved: {name}.pdf / .eps / .png")

print("figure export ready")

figure export ready


In [30]:
# %%
# --- [FIG] RQ1: basic-info floor -> hetero-GNN uplift ------------------------
# Reads saved tables (02, 03); no retraining. Mask A, test split.
import numpy as np, pandas as pd, matplotlib.pyplot as plt

TABLES = ROOT / "artifacts" / "tables"
floor = pd.read_csv(TABLES / "02_baseline_basic.csv")
gnn = pd.read_csv(TABLES / "03_gnn_reproduce.csv")

f_at = floor[(floor["mask"] == "A") & (floor["split"] == "test")]
floor_auc = f_at["AUC"].max()
floor_model = f_at.loc[f_at["AUC"].idxmax(), "model"]
gnn_auc = gnn[(gnn["mask"] == "A") & (gnn["split"] == "test")]["AUC"].iloc[0]

fig, ax = plt.subplots(figsize=(4.6, 4.2))
x = np.arange(2)
ax.bar(x, [floor_auc, gnn_auc], width=0.55,
       color=["#9aa0a6", "#c0392b"], edgecolor="black", linewidth=0.8)
for xi, v in zip(x, [floor_auc, gnn_auc]):
    ax.text(xi, v + 0.008, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
ax.annotate("", xy=(1, gnn_auc), xytext=(1, floor_auc),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.2))
ax.text(1.28, (floor_auc + gnn_auc) / 2, f"+{gnn_auc - floor_auc:.3f}",
        ha="left", va="center", fontsize=10, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([f"Basic-info floor\n({floor_model})",
                    "Hetero-GNN\n(lawsuit + relations)"], fontsize=9)
ax.set_ylabel("Test AUC (mask A)", fontsize=10)
ax.set_ylim(0.50, 0.85)
ax.grid(axis="y", ls=":", alpha=0.5)
for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
plt.tight_layout()
save_fig(fig, "fig_rq1_uplift")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


saved: fig_rq1_uplift.pdf / .eps / .png


In [31]:
# %%
# --- [FIG] Contrast cases: intra wrong -> contagion right --------------------
import numpy as np, matplotlib.pyplot as plt

topk = contrast_df.head(8)
fig, ax = plt.subplots(figsize=(6.0, 4.5))
yp = np.arange(len(topk))
ax.barh(yp - 0.2, topk["S0_prob"], height=0.4, color="#9aa0a6",
        edgecolor="black", linewidth=0.6, label="Intra-only")
ax.barh(yp + 0.2, topk["S1_prob"], height=0.4, color="#c0392b",
        edgecolor="black", linewidth=0.6, label="+Contagion")
ax.axvline(0.5, ls="--", c="k", lw=1)
ax.set_yticks(yp); ax.set_yticklabels(topk["name"], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Predicted bankruptcy probability", fontsize=10)
ax.set_xlim(0, 1)
ax.legend(loc="lower right", fontsize=9, frameon=False)
for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
plt.tight_layout()
save_fig(fig, "fig_contrast_cases")
plt.close(fig)

saved: fig_contrast_cases.pdf / .eps / .png


In [32]:
# %%
# --- [FIG] Relational neighborhood network (alpha-free for EPS) --------------
import networkx as nx, matplotlib.pyplot as plt

yv = comp.y.numpy()
G = nx.MultiDiGraph()
for n in nodes:
    is_comp = n in g2l
    lab = None
    if is_comp:
        li = g2l[n]
        lab = int(yv[li]) if (comp.train_mask[li] or comp.val_mask[li]
                              or comp.test_mask[li]) else -1
    G.add_node(n, is_company=is_comp, label=lab)
for (s, d, r) in elist:
    G.add_edge(s, d, rel=REL_NAME.get(r, str(r)))

def ncolor(n):
    if not G.nodes[n]["is_company"]: return "#d9e6f2"
    return {1: "#e74c3c", 0: "#2ecc71"}.get(G.nodes[n]["label"], "#cccccc")

pos = nx.spring_layout(G, seed=42, k=0.7)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#bbbbbb", arrows=False, width=0.9)
nx.draw_networkx_nodes(
    G, pos, ax=ax, node_color=[ncolor(n) for n in G.nodes],
    node_size=[560 if n == center else (260 if n in g2l else 130) for n in G.nodes],
    edgecolors=["black" if n == center else "#555555" for n in G.nodes],
    linewidths=[2.0 if n == center else 0.5 for n in G.nodes])
nx.draw_networkx_labels(G, pos, ax=ax,
    labels={n: label_for(n, n in g2l) for n in G.nodes}, font_size=7)
elabels = {}
for u, v, dd in G.edges(data=True):
    elabels.setdefault((u, v), dd["rel"])
nx.draw_networkx_edge_labels(G, pos, ax=ax, edge_labels=elabels, font_size=5.5)
ax.axis("off")
plt.tight_layout()
save_fig(fig, f"fig_network_{center}")
plt.close(fig)

saved: fig_network_1562.pdf / .eps / .png
